#Practica 6:
Aprovechando el dataframe de semestre (enero-junio 2020), cargar las tablas del pdf "Tablas Telefonia.pdf", hacer merge con las 3 tablas que contiene el pdf, para obtener la dimension franja, la dimensión Descripción, promociones a aplicar, aplicar el descuento al importe factura según su zona.
- Primero aplicar el descuento por Zona (lo tenemos en la tabla "Promociones a aplicar")
- Guardar en un csv, la suma de importe por Franja.
- Guardar en un csv, el numero de facturas por descripción de la satisfacción.

In [4]:
# Cargar las tablas de un pdf
# 0. Importar las librerias

import pdfplumber
import pandas as pd

# 1. Ajustar la ruta del archivo
ruta_fichero = "/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/Datos Telefonia Separados Meses Comerciales URL.xlsx"

# 2. Descargar los dataframes de ene-jun + satisfaccion + comerciales
fras_enero = pd.read_excel(ruta_fichero,sheet_name="Facturación Enero 2020", header=2)

fras_febrero = pd.read_excel(ruta_fichero,sheet_name="Facturación Febrero 2020")

fras_marzo = pd.read_excel(ruta_fichero,sheet_name="Facturación Marzo 2020",header=1)

fras_abril = pd.read_excel(ruta_fichero,sheet_name="Facturación Abril 2020",header=1)

fras_mayo = pd.read_excel(ruta_fichero,sheet_name="Facturación Mayo 2020",header=1)

fras_junio = pd.read_excel(ruta_fichero,sheet_name="Facturación Junio 2020",header=2)



# 3. Concatenar los meses
fras_semestre = pd.concat([fras_enero, fras_febrero, fras_marzo, fras_abril, fras_mayo, fras_junio], ignore_index=True)

# 4 comprobar si el dataframe esta bien

fras_semestre

# 5. Eliminar filas en blanco

fras_semestre = fras_semestre.dropna(how='all')

# 6 cargar dataframe localidades necesario para calculo descuento por zonas

localidades = pd.read_excel(ruta_fichero,sheet_name="Localidad", header=0)

# 7 Dividir la columna por un caracteer, quedarnos con la segunda columna, indice 1
# Quiltar los espacios en blanco

localidades['Localidad'] = localidades['Localidad'].str.split("#").str[1].str.strip()
localidades

# 8 Dejar solo 2 columnas.  (no necesitamos mas)

localidades = localidades[['Localidad', 'Zona']]
localidades

# 9. Comprobar el dataframe localidades

localidades

# 10. Merge df semestre con localidades

fras_semestre = fras_semestre.merge(localidades, on='Localidad', how='inner')

fras_semestre

# 11. Ajustar la ruta del fichero

ruta_pdf = '/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/Tablas Telefonía.pdf'

# 12. Crear una lista para almacenar las tablas extraidas

tablas =[]

# 13. Vamos a crear un bucle para ir cargando las tablas, miraremos las diferentes tablas
#  de las diferentes paginas

with pdfplumber.open(ruta_pdf) as pdf:
    for page in pdf.pages:          # con esto indicamos que recorra todas las paginas del pdf
        # Extraer las tablas de la pagina que estoy recorriendo
        tablas_extraidas = page.extract_tables()
        
        # Vamos a recorrer la coleccion de tablas extraibles 
        for tabla in tablas_extraidas:
            df = pd.DataFrame(tabla)
            tablas.append(df)

# 14 y 15. Acceder al dataframe resultante despues de asignar a un objeto

len(tablas) # Para saber cuantas tablas hay en la lista

# Lo mas optimo, es clonar el dataframe en un objeto para tenerlo separado 
franjas = tablas[0]
franjas

# 16. elimnar las columnas no validas
franjas = franjas.drop([1,2,4,5,7,8],axis=1).reset_index(drop=True)

# 17. Eliminar la primera fila
franjas = franjas.drop([0], axis=0).reset_index(drop=True)

# 18. Poner nombre a las columnas
franjas = franjas.rename(columns={
    0:"Decada",
    3:"Franja",
    6:"Descripcion"
})

# 19. Ajustar campo decada como numero entero

franjas['Decada'] = franjas['Decada'].astype('Int64')

# 20. Ajustar dataframe de descripciones

descripcion = tablas[1]

# 21. Eliminamos columnas en blanco

descripcion = descripcion.drop([1,2,4,5],axis=1).reset_index(drop=True)

# 22. Eliminar la primera fila

descripcion = descripcion.drop([0], axis=0).reset_index(drop=True)

# 23. Renombramos las columnas

descripcion = descripcion.rename(columns={
    0:"Satisfacción Cliente",
    3:"Descripción Satisfacción"
})

# 24. Convertir a entero el campo Satisfaccion cliente

descripcion['Satisfacción Cliente'] = descripcion['Satisfacción Cliente'].astype('Int64')

# 25. Crear dataframe promociones

promociones = tablas[2]

# 26. Eliminar las columnas sobrantes

promociones = promociones.drop([1,2,4,5,7,8],axis=1).reset_index(drop=True)

# 27. Eliminar la primera fila

promociones = promociones.drop([0], axis=0).reset_index(drop=True)

# 28. Renombrar las columnas

promociones = promociones.rename(columns={
    0:"Descripción promoción",
    3:"Aplicar a.",
    6:"% Dto."
})

# 29. Ajustar el % de descuento para poder calcular

promociones["% Dto."] = (
    promociones['% Dto.']
    .str.replace("%","")
    .str.replace(",",".")
    .astype(float) / 100
)


# 30. Calcular la edad en el dataframe de semestre, ver el tipo de datos del Dataframe

#fras_semestre.info()

# 31.1 CAnviar tipo de dato 

fras_semestre['Fecha Nacimiento'] = pd.to_datetime(fras_semestre['Fecha Nacimiento'])

# 31.2 Calcular el campo edad

fras_semestre['Edad'] = ((pd.to_datetime('today') - fras_semestre['Fecha Nacimiento']).dt.days // 365).astype("Int64")

# 31.3 Calcular la decada + convertir a integer

fras_semestre['Decada'] = ((fras_semestre["Edad"] // 10) * 10).astype("Int64")  # // para que solo coja parte entera de 56 50 o 5

# 32. Hacer Merge de fras_semestre con franjas (todos los merge anidadeos) no ponemos inner JOIN porque ya es por defecto

fras_semestre = (fras_semestre
                 .merge(franjas, on='Decada')
                 .merge(descripcion, left_on="Satisfacción", right_on="Satisfacción Cliente")
                 .merge(promociones, left_on="Zona", right_on="Aplicar a.")
                 )

# 33. Calculo del importe neto aplicado el descuento por zona

fras_semestre['Importe factura neto'] = round(fras_semestre['Importe factura'] * (1-fras_semestre["% Dto."]),2)

# 34. Quedarme con los campos que necesito para el calculo

resultado = fras_semestre[['Importe factura neto','Franja','Descripción Satisfacción']]

# 35. Agrupar el importe fra neto por franjas

importe_franjas = resultado.groupby(['Franja'])['Importe factura neto'].sum().reset_index()

# 36. Agrupar el numero de facturas por descripcion por la satisfaccion

facturas_descripcion = resultado.groupby(['Descripción Satisfacción'])['Importe factura neto'].count().reset_index(name='Numero fras.')

# 37. Guardar a CSV 

importe_franjas.to_csv('/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Importe franjas.csv')


facturas_descripcion.to_csv('/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/facturas_descripcion.csv')


print("terminado con exito")




terminado con exito


Formularios en Pyrhon con modulo tkinter (es nativo biene instalado con Python, no ay que instalar)
- Un formulario es una interficie grafica de escritorio, es una ventana donde podremos colocar widgets.
- Widgets principales:
- Label = Etiqueta
- Entry = Caja de texto donde (el usuario puede escribir)
- Button = Boton para ejecutar
- Checkbutton / Radiobutton Casillas de verificacion

Como se estructura un formulario:
- Ventana raiz (tk) = es el contenedor principal
- Gestor de geometria, se usa para posicionar elementos
- Bucle principal o mainloop

In [2]:
# 0. Importar las librerias

import tkinter as tk

# 1. Crear la ventana principal
root = tk.Tk()

root.title("DAshboards de KPIs")

# 2. Vamos a ajustar las medidas de la ventana. en pixels, 2 ultimos valores ajusta left desde izda 100 y el top desde arriba 50

root.geometry("500x300+100+50")

# 3. Bucle principal mantiene la ventana viva y también escucha eventos

root.mainloop()

: 